# PySpark 6: structured streaming

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-06-structured-streaming.ipynb)

Orders and clickstream events now arrive continuously. Finance still wants each event counted once, and the
job must resume after a crash without reprocessing or losing data.

This module runs real Structured Streaming queries over local files: incremental processing with a
checkpoint, deduplication with a watermark, output modes, event-time windows, a stream-static join, and
query progress metrics. It then covers Kafka, custom per-key state, and recovery after a failed write.

Each section explains how the feature works, where it matters and what to watch out for, then ends with a runnable example. Run it, read the output, and compare it with the notes on what to notice. Run the cells in order.

**Prerequisites:** [PySpark 5: writing, pipelines and quality](pyspark-05-writing-pipelines-and-quality.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-foundations.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

In [ ]:
import json


def land_json(directory, name, rows, mtime):
    """Write one JSON-lines file with a fixed modification time, so files are processed in order."""
    os.makedirs(directory, exist_ok=True)
    path = os.path.join(directory, name)
    with open(path, "w") as handle:
        handle.write("\n".join(json.dumps(row) for row in rows))
    os.utime(path, (mtime, mtime))
    return path


def run_until_done(query, timeout=60):
    """Wait for an availableNow query to finish, and always stop it."""
    try:
        if not query.awaitTermination(timeout):
            raise TimeoutError("stream did not finish")
    finally:
        query.stop()
    return query

## Process arriving files with a checkpoint

A streaming query is a batch plan that Spark re-runs on each **trigger** over the data that arrived since the
last one (a **micro-batch**). The **checkpoint** directory records which input has been processed and the
state of stateful operators, so a restarted query continues where it stopped.

`foreachBatch` hands each micro-batch to your function as an ordinary DataFrame, so any batch write, including
a `MERGE`, can be the sink. `trigger(availableNow=True)` processes everything available and stops: the
scheduled-batch pattern with streaming bookkeeping. `dropDuplicatesWithinWatermark` removes repeats of the
same key across micro-batches while the watermark bounds how long keys are remembered.

**Where it matters**
- **Incremental file ingestion:** Auto Loader on Databricks (`format("cloudFiles")`), or the plain file source elsewhere.
- **Hourly jobs** that should process only new files without tracking them yourself.

**Watch out for**
- One checkpoint per query, never shared, never moved. A new checkpoint path starts a new history and reprocesses everything.
- `foreachBatch` can receive the same batch again after a failure. Make the write idempotent.
- Never `collect()` a micro-batch to the driver inside `foreachBatch`; keep it distributed.

### Example

Land two files (the second repeats event `e2`), stream them one file per micro-batch, deduplicate within a
30-minute watermark, and write each batch to its own folder. Run the query again with the same checkpoint:
nothing is reprocessed.

In [ ]:
demo_dir = tempfile.mkdtemp(prefix="stream-dedup-", dir=WORK_DIR)
landing_dir = os.path.join(demo_dir, "landing")
checkpoint_dir = os.path.join(demo_dir, "checkpoints", "events_silver")
silver_dir = os.path.join(demo_dir, "silver")
EVENT_SCHEMA = "event_id string, user_id int, event_ts timestamp, event_type string"

land_json(landing_dir, "part-0.json", [
    {"event_id": "e1", "user_id": 1, "event_ts": "2026-01-06T09:00:00", "event_type": "view"},
    {"event_id": "e2", "user_id": 1, "event_ts": "2026-01-06T09:02:00", "event_type": "add_to_cart"},
], mtime=1_000)
land_json(landing_dir, "part-1.json", [
    {"event_id": "e2", "user_id": 1, "event_ts": "2026-01-06T09:02:00", "event_type": "add_to_cart"},
    {"event_id": "e3", "user_id": 2, "event_ts": "2026-01-06T09:05:00", "event_type": "purchase"},
], mtime=2_000)

batch_ids = []


def write_batch(batch_df, batch_id):
    batch_ids.append(batch_id)
    # Idempotent per batch: a retried batch overwrites its own folder instead of appending twice.
    batch_df.write.mode("overwrite").parquet(os.path.join(silver_dir, f"batch_id={batch_id}"))


def run_dedup_stream():
    query = (
        spark.readStream.schema(EVENT_SCHEMA)
        .option("maxFilesPerTrigger", 1)
        .json(landing_dir)
        .withWatermark("event_ts", "30 minutes")
        .dropDuplicatesWithinWatermark(["event_id"])
        .writeStream.foreachBatch(write_batch)
        .option("checkpointLocation", checkpoint_dir)
        .trigger(availableNow=True)
        .start()
    )
    run_until_done(query)


def silver_event_ids():
    return sorted(r.event_id for r in spark.read.parquet(silver_dir).select("event_id").collect())

run_dedup_stream()
print("first run  -> batches:", batch_ids, "| silver events:", silver_event_ids())

run_dedup_stream()  # same checkpoint: both files are already committed
print("second run -> batches:", batch_ids, "| silver events:", silver_event_ids())

**What to notice:** The first run processes one file per micro-batch (batches 0 and 1), and `e2`, repeated in the second file,
is written once. Batch 2 is a no-data batch: Spark runs it to advance the watermark and clean up
deduplication state. The second run, on the same checkpoint, adds no batch at all: the checkpoint already
records both files as processed, so nothing is read twice.

## Choose an output mode and an event-time window

`F.window("event_ts", "5 minutes")` groups events into tumbling windows by **event time**, the time in the
data, not arrival time. A **watermark** (`withWatermark("event_ts", "10 minutes")`) tells Spark how late data
may arrive: once the latest event time seen minus 10 minutes passes a window's end, that window is final and
its state can be dropped. Sliding windows take a slide interval, and `session_window` groups bursts of
activity separated by a gap.

The **output mode** decides what the sink receives on each trigger:

| mode | emits | needs |
| --- | --- | --- |
| `append` | only rows that will never change | a watermark for aggregations; rows appear after the window closes |
| `update` | rows that changed in this trigger | works with or without a watermark |
| `complete` | the whole result every time | aggregations only; state is never dropped |

**Where it matters**
- **Minute-by-minute dashboards:** `update` mode into a table keyed by window.
- **Final, immutable aggregates** written to files: `append` with a watermark.

**Watch out for**
- `complete` mode keeps every window forever; state grows without bound on a long-running stream.
- A windowed aggregation without a watermark never drops state.
- In `append` mode results appear only after the watermark passes the window end, so latency is at least the watermark delay.

### Example

Stream two files into five-minute windows. The second file adds an event to the first window and opens a new
one. `update` mode receives the first window twice (count 2, then 3); `complete` mode ends with the full
result. The progress metrics confirm that four input rows were processed.

In [ ]:
window_dir = os.path.join(WORK_DIR, "stream-windows")
land_json(os.path.join(window_dir, "input"), "a.json",
          [{"ts": "2026-01-01 10:01:00", "amount": 10}, {"ts": "2026-01-01 10:02:00", "amount": 20}], mtime=1_000)
land_json(os.path.join(window_dir, "input"), "b.json",
          [{"ts": "2026-01-01 10:03:00", "amount": 5}, {"ts": "2026-01-01 10:07:00", "amount": 7}], mtime=2_000)


def run_windows(mode):
    totals = (
        spark.readStream.schema("ts timestamp, amount int")
        .option("maxFilesPerTrigger", 1)
        .json(os.path.join(window_dir, "input"))
        .withWatermark("ts", "10 minutes")
        .groupBy(F.window("ts", "5 minutes"))
        .agg(F.count("*").alias("events"), F.sum("amount").alias("amount"))
    )
    query = (
        totals.writeStream.format("memory").queryName(f"windows_{mode}").outputMode(mode)
        .option("checkpointLocation", os.path.join(window_dir, f"checkpoint-{mode}"))
        .trigger(availableNow=True).start()
    )
    run_until_done(query)
    rows = spark.table(f"windows_{mode}").collect()
    return sorted((r.window.start.strftime("%H:%M"), r.events) for r in rows), query

update_rows, update_query = run_windows("update")
complete_rows, _ = run_windows("complete")
print("update   (window start, events):", update_rows)
print("complete (window start, events):", complete_rows)

for progress in update_query.recentProgress:  # one entry per micro-batch
    p = progress if isinstance(progress, dict) else progress.jsonValue()
    print(f"batch {p['batchId']}: {p['numInputRows']} input rows")

**What to notice:** In `update` mode the sink received the 10:00 window twice: with 2 events after the first file and 3 after
the second, when the 10:03 event arrived. `complete` mode ends with the whole current result, one row per
window. The progress entries show how many rows each micro-batch read, the same numbers you watch on a
production stream; the last, empty batch only advances the watermark.

## Join a stream with a static table

A **stream-static join** enriches each micro-batch with a batch DataFrame, such as a customer dimension.
Spark re-reads the static side on every trigger, so a Delta dimension that changes is picked up by the next
micro-batch. Inner and left joins with the stream on the left are supported. A **stream-stream join**
(orders with payments) needs a watermark on both sides and a time-range condition so Spark can drop
buffered rows.

**Where it matters**
- **Enriching events** with customer region or product category before writing silver.
- **Filtering** a stream to active accounts kept in a table.

**Watch out for**
- A large static side is re-read every trigger. Keep it small, or broadcast it explicitly.
- A stream-stream join without watermarks buffers every row forever.

### Example

Enrich streamed events with the customer region. User 9 has no customer row and is kept by the left join.

In [ ]:
join_dir = os.path.join(WORK_DIR, "stream-join")
land_json(os.path.join(join_dir, "input"), "events.json", [
    {"event_id": "j1", "user_id": 1, "event_ts": "2026-01-06T09:00:00", "event_type": "view"},
    {"event_id": "j2", "user_id": 3, "event_ts": "2026-01-06T09:01:00", "event_type": "view"},
    {"event_id": "j3", "user_id": 9, "event_ts": "2026-01-06T09:02:00", "event_type": "view"},
], mtime=1_000)

regions = customers.select(F.col("customer_id").alias("user_id"), "region")
enriched = spark.readStream.schema(EVENT_SCHEMA).json(os.path.join(join_dir, "input")).join(regions, "user_id", "left")
query = (
    enriched.writeStream.format("memory").queryName("enriched_events").outputMode("append")
    .option("checkpointLocation", os.path.join(join_dir, "checkpoint"))
    .trigger(availableNow=True).start()
)
run_until_done(query)

spark.table("enriched_events").orderBy("event_id").show()

**What to notice:** Each streamed event picked up its customer's region from the static table. User 9 has no customer row; the
`left` join keeps the event with a null region instead of dropping it.

## Read from Kafka

The Kafka source reads topics as rows of `key`, `value` (binary), `topic`, `partition`, `offset` and
`timestamp`. Decode `value` with `from_json` and a schema, keep the topic, partition and offset for lineage,
and bound each micro-batch with `maxOffsetsPerTrigger`. The checkpoint stores the offsets, so `startingOffsets`
applies only to a brand-new query.

```python
orders_stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")      # first run only; the checkpoint wins afterwards
    .option("maxOffsetsPerTrigger", 500_000)    # bound each micro-batch
    .load()
    .select(
        F.from_json(F.col("value").cast("string"), order_schema).alias("o"),
        "topic", "partition", "offset",
    )
    .select("o.*", "topic", "partition", "offset")
)
(
    orders_stream.writeStream
    .foreachBatch(merge_into_silver)             # idempotent MERGE per batch
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="1 minute")
    .start()
)
```

The Kafka connector is a separate package (`spark-sql-kafka-0-10`) and needs a broker, so this is shown, not
run.

**Where it matters**
- **Event pipelines** from application services into a lakehouse table.
- **CDC** from Debezium topics into silver tables with a merge per batch.

**Watch out for**
- Without `maxOffsetsPerTrigger`, a backlog becomes one enormous first batch.
- Malformed messages make `from_json` return null. Route them to a quarantine with topic, partition and offset so they can be replayed.
- Kafka retention shorter than your longest outage means offsets in the checkpoint may no longer exist.

### Example

Size a trigger: a topic receives 2,000 events a second and each micro-batch should cover at most a minute of
traffic. That is `maxOffsetsPerTrigger = 2,000 x 60 = 120,000`, split across the topic's partitions. If a
batch takes longer than the trigger interval, batches start late and latency climbs; raise resources or the
interval, not the cap.

## Keep and expire custom per-key state

Built-in stateful operators (windowed aggregation, `dropDuplicatesWithinWatermark`, stream-stream joins) cover
most pipelines. When the rule is your own (a session that closes after 30 idle minutes, "alert on the third
failed payment"), Spark 4 adds `transformWithStateInPandas`. It replaces `applyInPandasWithState` as the
arbitrary-state API.

You write a `StatefulProcessor`. `init` declares named state variables (`ValueState`, `ListState`,
`MapState`), each with an optional TTL; `handleInputRows` runs once per key per micro-batch; timers fire
`handleExpiredTimer`.

```python
from typing import Iterator

import pandas as pd
from pyspark.sql.streaming import StatefulProcessor, StatefulProcessorHandle

spark.conf.set(
    "spark.sql.streaming.stateStore.providerClass",
    "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider",
)


class EventCounter(StatefulProcessor):
    def init(self, handle: StatefulProcessorHandle) -> None:
        self.seen = handle.getValueState("seen", "n long")  # one value per user_id

    def handleInputRows(self, key, rows, timerValues) -> Iterator[pd.DataFrame]:
        total = self.seen.get()[0] if self.seen.exists() else 0
        for batch in rows:  # pandas DataFrames holding this key's new rows
            total += len(batch)
        self.seen.update((total,))
        yield pd.DataFrame({"user_id": [key[0]], "events_seen": [total]})

    def close(self) -> None:
        pass


counts = events_stream.groupBy("user_id").transformWithStateInPandas(
    statefulProcessor=EventCounter(),
    outputStructType="user_id string, events_seen long",
    outputMode="Update",
    timeMode="None",  # "ProcessingTime" or "EventTime" to register timers
)
```

It needs the RocksDB state store and `pandas`, `pyarrow` and `protobuf` on the workers, so it is shown, not
run.

**Where it matters**
- **Sessionization** with custom timeout rules.
- **Fraud and alerting rules** that count events per account over time.

**Watch out for**
- State with no TTL or timer: a key that never returns holds its state forever, and checkpoints slow down.
- Changing the state schema can make an existing checkpoint unusable. Plan state changes like table migrations.

### Example

Estimate state size before shipping. 5 million active users with a 200-byte state each is about 1 GB of
state, written to the checkpoint on every trigger. A 24-hour TTL on users who go quiet keeps it from growing
with every user ever seen.

## Recover a streaming query after a failed write

On restart with the same checkpoint, Spark replays any micro-batch that was not recorded as committed. With
`foreachBatch` that means your function can receive the same `batch_id` and the same rows again. The sink
must make that harmless: overwrite a location owned by the batch, merge by a stable key and source version,
or record the batch id in the same transaction as the output.

**Where it matters**
- **Driver crashes or cluster restarts** between the sink write and the checkpoint commit.
- **Deployments:** a new code version restarts the query on the old checkpoint.

**Watch out for**
- Changing the source, stateful operators or the state schema may make the old checkpoint incompatible. Check before deploying.
- A `batch_id` is only unique within one checkpoint's history. Do not use it as a global key.

### Example

Simulate a replay: call the batch function twice with the same id and rows, as Spark would after a crash.
The batch's folder still holds each event once.

In [ ]:
replayed = spark.createDataFrame(
    [("e9", 3, datetime(2026, 1, 6, 9, 30), "view"), ("e10", 3, datetime(2026, 1, 6, 9, 31), "view")],
    EVENT_SCHEMA,
)
write_batch(replayed, 99)
write_batch(replayed, 99)  # the same batch delivered again after a restart

spark.read.parquet(os.path.join(silver_dir, "batch_id=99")).orderBy("event_id").show()

**What to notice:** Batch 99 was written twice, but its folder holds each event once: the second write overwrote the first. A
plain append in `foreachBatch` would have stored four rows.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** What does the checkpoint store, and what happens if you change its path?

**A.** Which input has been processed (offsets or files), committed batches, and operator state. A new path starts a new query history, so all input is processed again and state is lost.

**Q2.** A watermark is 10 minutes and the latest event time seen is 10:40. Which windows can be finalized?

**A.** Windows that end at or before 10:30, the watermark. Their state can be dropped and, in append mode, they are emitted.

**Q3.** When would you use `update` mode instead of `complete`?

**A.** Almost always for aggregations on a long-running stream: `update` emits only changed rows and lets the watermark drop old state, while `complete` re-emits and keeps every window forever.

**Q4.** Why must a `foreachBatch` function be idempotent?

**A.** After a failure Spark can call it again with the same batch id and rows. A plain append would write them twice.

**Q5.** What does `trigger(availableNow=True)` give you over a plain batch job?

**A.** It processes all new input and stops, but keeps the streaming bookkeeping: the checkpoint remembers what was processed, so you never track files or offsets yourself.

**Q6.** A stream-stream join's state keeps growing. What is missing?

**A.** Watermarks on both inputs and a time-range join condition, so Spark knows when a buffered row can no longer match.

## Key takeaways

- A streaming query is a batch plan re-run per trigger; the checkpoint makes it restartable. One per query.
- A watermark bounds lateness and state; without one, state grows forever.
- Use `update` or `append` for long-running aggregations; `complete` keeps everything.
- Make `foreachBatch` writes idempotent: overwrite what the batch owns or merge by key and version.
- Bound Kafka micro-batches with `maxOffsetsPerTrigger`, and keep topic, partition and offset for lineage.

**Next:** [PySpark 7: monitoring, performance and tuning](pyspark-07-monitoring-and-performance.ipynb)